In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from collections import defaultdict
import os, tempfile
from zipfile import ZipFile
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
# XML parser: Extracts shapes from an XML file.
# Each shape is represented as a list of (x, y) coordinates.
# -------------------------------------------------------------------

class parseXMLfile():
    def __init__(self, file: str, n_coordinates_min: int, verbose: int = 0):
        self._input_floc = file
        self._verbose = verbose
        self._n_coordinates_min = n_coordinates_min
        self._dict_shapes = {}
        self._skipped_shaped = 0

    def _read_file(self):
        tree = ET.parse(self._input_floc)
        root = tree.getroot()
        
        # Iterate through the XML tree to find elements representing shapes
        for shape in root:
            if not shape.tag.startswith('Shape_'):
                continue
            
            coordinates = []
            point_count = int(shape.find('PointCount').text)
            
            for i in range(1, point_count + 1):
                x_tag = f'X_{i}'
                y_tag = f'Y_{i}'
                x_value = int(shape.find(x_tag).text)
                y_value = int(shape.find(y_tag).text)
                coordinates.append((x_value, y_value))
            
            # Only keep shapes with at least n_coordinates_min points
            if len(coordinates) < self._n_coordinates_min:
                self._skipped_shaped += 1
                continue
            
            self._dict_shapes[shape.tag] = coordinates

    @property
    def shapes(self):
        self._read_file()
        return self._dict_shapes, self._skipped_shaped



In [4]:
# CentroidSelector: processes polygons to extract centroids,
# groups them by niche, and plots them with a distinct color per group.
# -------------------------------------------------------------------

class CentroidSelector():
    def __init__(self, dict_polygons: dict, dict_state: dict):
        """
        dict_polygons: a dictionary with keys as polygon identifiers and values as dicts
                       containing at least a 'centroid' and 'niche' key.
        dict_state: a state dictionary to hold cluster centroids.
        """
        self._dict_polygons = dict_polygons
        self._dict_state = dict_state
        self._dict_polygon_centroids = self._get_polygon_centroids()
        # Use the provided cluster centroid structure from state.
        self._dict_cluster_centroids = self._dict_state.get('cl_centroids', {
            'idx': [],
            'x': [],
            'y': [],
            'color': [],
            'size': [],
            'label': []
        })

    def _get_polygon_centroids(self) -> dict:
        """
        Groups centroids by niche and assigns each group a unique color.
        Returns a dictionary keyed by niche, where each value is a dict 
        with lists for x, y, color, size, and label.
        """
        # First, get all unique niche names
        unique_niches = { poly_data.get('niche', 'Unknown') 
                          for poly_data in self._dict_polygons.values() }
        unique_niches = sorted(unique_niches)
        num_niches = len(unique_niches)
        cmap = plt.get_cmap('tab20', num_niches)
        color_dict = {niche: cmap(i) for i, niche in enumerate(unique_niches)}
        
        niche_groups = {}
        for key, poly_data in self._dict_polygons.items():
            niche = poly_data.get('niche', 'Unknown')
            x, y = poly_data['centroid']
            if niche not in niche_groups:
                niche_groups[niche] = {'x': [], 'y': [], 'color': [], 'size': [], 'label': []}
            niche_groups[niche]['x'].append(x)
            niche_groups[niche]['y'].append(y)
            niche_groups[niche]['color'].append(color_dict[niche])
            niche_groups[niche]['size'].append(3)  # smaller marker size
            niche_groups[niche]['label'].append(key)
        return niche_groups

    def render_centroids1(self):
        print("Rendering centroids plot ...")
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.grid(color='#41b6c4', linestyle='dotted', linewidth=0.5)
        
        # Plot polygon centroids, grouping by niche and using the group-specific color.
        for niche, data in self._dict_polygon_centroids.items():
            ax.scatter(data['x'], data['y'], c=data['color'], s=data['size'], label=niche)
        
        # Plot cluster centroids (if any exist) with a distinct marker.
        if self._dict_cluster_centroids['x']:
            ax.scatter(self._dict_cluster_centroids['x'], self._dict_cluster_centroids['y'],
                       c=self._dict_cluster_centroids['color'], s=self._dict_cluster_centroids['size'],
                       label='Cluster Centroids', marker='^')
        
        ax.ticklabel_format(useOffset=False, style='plain')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        # Adjust layout to make space for the legend
        fig.tight_layout(rect=[0, 0, 1.5, 1])
        plt.show()

    def render_centroids(self, ax):
        for niche, data in self._dict_polygon_centroids.items():
            ax.scatter(data['x'], data['y'], c=data['color'], s=data['size'], label=niche, alpha=0.7)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.set_title("Niche Distribution")
        ax.grid(color='#41b6c4', linestyle='dotted', linewidth=0.5)

    def _update_cl_centroid(self, idx, x_coord, y_coord):
        cl_idx = len(self._dict_cluster_centroids['color'])
        self._dict_cluster_centroids['idx'].append(idx)
        self._dict_cluster_centroids['x'].append(int(x_coord))
        self._dict_cluster_centroids['y'].append(int(y_coord))
        self._dict_cluster_centroids['color'].append('#525252')
        self._dict_cluster_centroids['size'].append(150)
        self._dict_cluster_centroids['label'].append(f'Cluster {cl_idx}')
        print(f'Added cluster centroid: {idx} -> ({x_coord}, {y_coord})')

    def reset_cl_centroid(self):
        self._dict_cluster_centroids['idx'] = []
        self._dict_cluster_centroids['x'] = []
        self._dict_cluster_centroids['y'] = []
        self._dict_cluster_centroids['color'] = []
        self._dict_cluster_centroids['size'] = []
        self._dict_cluster_centroids['label'] = []
                
    @property
    def get_centroids(self):
        return self._dict_cluster_centroids

# compute arithmetic centroid from a list of (x, y)
def compute_centroid(coordinates):
    if not coordinates:
        return (0, 0)
    x_coords = [pt[0] for pt in coordinates]
    y_coords = [pt[1] for pt in coordinates]
    return (sum(x_coords) / len(coordinates), sum(y_coords) / len(coordinates))

In [5]:
# Spatial niche annotations: Reads all XML files for one slide and combines shapes with defined niche annotations.
# -------------------------------------------------------------------

def parse_spatial_niches(base_path, base_filename, n_coordinates_min=3, verbose=1):
    base_path = os.path.join(base_path, base_filename)
    polygons = {}
    niche_files = [f for f in os.listdir(base_path) if f.startswith(f"{base_filename}_Niche") and f.endswith(".xml")]
    num_niches = len(niche_files)
    
    xml_file_all = os.path.join(base_path, f"{base_filename}_All.xml")
    parser_all = parseXMLfile(file=xml_file_all, n_coordinates_min=n_coordinates_min, verbose=verbose)
    shapes_all, skipped_all = parser_all.shapes
    print(f"All: Parsed {len(shapes_all)} shapes; skipped {skipped_all} shapes.")
    for shape_name, coords in shapes_all.items():
        centroid = compute_centroid(coords)
        polygons[f"All_{shape_name}"] = {'centroid': centroid, 'coordinates': coords, 'niche': "All"}
    
    for niche_file in niche_files:
        niche_id = niche_file.replace(f"{base_filename}_Niche", "").replace(".xml", "")
        xml_file = os.path.join(base_path, niche_file)
        parser = parseXMLfile(file=xml_file, n_coordinates_min=n_coordinates_min, verbose=verbose)
        shapes, skipped = parser.shapes
        print(f"Niche {niche_id}: Parsed {len(shapes)} shapes; skipped {skipped} shapes.")
        for shape_name, coords in shapes.items():
            centroid = compute_centroid(coords)
            polygons[f"Niche_{niche_id}_{shape_name}"] = {'centroid': centroid, 'coordinates': coords, 'niche': f"Niche_{niche_id}"}
    
    return polygons

In [6]:
def plot_polygons(polygons):
    state = {'cl_centroids': {'idx': [], 'x': [], 'y': [], 'color': [], 'size': [], 'label': []}}
    selector = CentroidSelector(polygons, state)
    selector.render_centroids()

In [7]:
# Luminal-to-periphery score: Sector-based calculation of the luminal to periphery score with
#   0 = closest shape to the luminal center
#   1 = most peripheral shape
# -------------------------------------------------------------------

def compute_lumina_periphery_scores(polygons, x_adjust=0, y_adjust=0, num_sectors=120):
    centroids = [data['centroid'] for data in polygons.values()]
    coords = np.array(centroids)

    x_center = np.mean(coords[:, 0]) + x_adjust
    y_center = np.mean(coords[:, 1]) + y_adjust

    distances = np.sqrt((coords[:, 0] - x_center)**2 + (coords[:, 1] - y_center)**2)
    relative_x = coords[:, 0] - x_center
    relative_y = coords[:, 1] - y_center
    angles = np.degrees(np.arctan2(relative_y, relative_x))
    angles[angles < 0] += 360

    sector_size = 360 / num_sectors
    scores = np.zeros(len(coords))

    for i in range(num_sectors):
        sector_start = i * sector_size
        sector_end = (i + 1) * sector_size
        sector_indices = np.where((angles >= sector_start) & (angles < sector_end))[0]
        if len(sector_indices) == 0:
            continue
        dists = distances[sector_indices]
        d_min, d_max = dists.min(), dists.max()
        if d_max == d_min:
            scores[sector_indices] = 0.5
        else:
            scores[sector_indices] = (dists - d_min) / (d_max - d_min)

    return scores, coords, (x_center, y_center)

In [8]:
# Plotting
def Lumina_Periphery_Score(polygons, x_adjust=0, y_adjust=0, num_sectors=120, ax=None):
    scores, coords, (x_center, y_center) = compute_lumina_periphery_scores(polygons, x_adjust, y_adjust, num_sectors)

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    sc = ax.scatter(coords[:, 0], coords[:, 1], c=scores, cmap='Reds', s=0.2)
    ax.scatter(x_center, y_center, marker='x', s=100, c='r')
    cbar = plt.colorbar(sc, ax=ax, orientation='vertical')
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Lumina', 'Periphery'])
    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate')
    ax.set_title('Lumina to Periphery Score')

In [9]:
# Calculating mean and median luminal-to-periphery scores
def calculate_niche_scores(polygons, ID, x_adjust=0, y_adjust=0, num_sectors=120):
    import pandas as pd
    scores, _, _ = compute_lumina_periphery_scores(polygons, x_adjust, y_adjust, num_sectors)
    keys = list(polygons.keys())
    polygon_scores = {keys[i]: scores[i] for i in range(len(keys))}

    result_rows = []
    for niche in set(data['niche'] for data in polygons.values()):
        if niche == "All":
            continue
        niche_keys = [k for k, v in polygons.items() if v['niche'] == niche]
        niche_scores = [polygon_scores[k] for k in niche_keys]
        result_rows.append({
            'Slide': ID,
            'Niche': niche,
            'MeanScore': np.mean(niche_scores),
            'MedianScore': np.median(niche_scores)
        })

    return pd.DataFrame(result_rows)

In [10]:
# Manual x/y offsets correct the estimated plaque center for each slide
slides_adjustment = [
    {'slide': 'EK1', 'x_offset': 250000, 'y_offset': 0,'sectors': 120},
    {'slide': 'EK2', 'x_offset': 85000, 'y_offset': 3000,'sectors': 120},
    {'slide': 'EK3', 'x_offset': 250000, 'y_offset': -80000,'sectors': 120},
    {'slide': 'EK4', 'x_offset': 190000, 'y_offset': -70000,'sectors': 120},
    {'slide': 'EK5', 'x_offset': 75000, 'y_offset': 75000,'sectors': 120}, 
    {'slide': 'EK6', 'x_offset': -60000, 'y_offset': -80000,'sectors': 120}, #-> separate
    {'slide': 'EK7', 'x_offset': 31000, 'y_offset': 110000,'sectors': 120},
    {'slide': 'EK8', 'x_offset': 130000, 'y_offset': -130000,'sectors': 120},
    {'slide': 'EK9', 'x_offset': 10000, 'y_offset': 130000,'sectors': 120},
    {'slide': 'EK10', 'x_offset': 80000, 'y_offset': -150000,'sectors': 120}, #-> separate
    {'slide': 'EK11', 'x_offset': 38000, 'y_offset': -120000, 'sectors': 90},
    {'slide': 'EK12', 'x_offset': 0, 'y_offset': 0,'sectors': 120},
    {'slide': 'EK13', 'x_offset': -100000, 'y_offset': -50000,'sectors': 120},
    {'slide': 'EK14', 'x_offset': 80000, 'y_offset': -10000,'sectors': 120},
    {'slide': 'EK15', 'x_offset': 80000, 'y_offset': -200000,'sectors': 120},
    {'slide': 'EK16', 'x_offset': -230000, 'y_offset': -150000,'sectors': 120},
    {'slide': 'EK17', 'x_offset': 80000, 'y_offset': 20000,'sectors': 120},
    {'slide': 'EK18', 'x_offset': 40000, 'y_offset': 170000,'sectors': 120},
    {'slide': 'EK19', 'x_offset': 60000, 'y_offset': -60000,'sectors': 120},
    {'slide': 'EK20', 'x_offset': 80000, 'y_offset': -20000,'sectors': 120},
    {'slide': 'EK21', 'x_offset': -130000, 'y_offset': -220000,'sectors': 120},
    {'slide': 'EK22', 'x_offset': 80000, 'y_offset': 000,'sectors': 120},
    {'slide': 'EK23', 'x_offset': 130000, 'y_offset': 0,'sectors': 120},
    {'slide': 'EK24', 'x_offset': -280000, 'y_offset': -80000,'sectors': 120},
]

In [13]:
all_stats = []

for slide_info in slides_adjustment:
    slide = parse_spatial_niches(
        base_path="./Atherosclerosis_Atlas/Proteomics/Spatial/XML/",
        base_filename=slide_info['slide'],
        verbose=1
    )
    niche_statistics = calculate_niche_scores(slide, ID=slide_info["slide"], x_adjust=slide_info['x_offset'], y_adjust=slide_info['y_offset'], num_sectors=slide_info['sectors'])
    all_stats.append(niche_statistics)

# Combine all dataframes
combined_df = pd.concat(all_stats, ignore_index=True)

# Save to CSV
combined_df.to_csv("Niche_statistics_luminal_periphery_scores.csv", index=False)
print("Saved combined niche statistics to niche_scores_summary.csv")


All: Parsed 28522 shapes; skipped 0 shapes.
Niche 1: Parsed 400 shapes; skipped 0 shapes.
Niche 10: Parsed 400 shapes; skipped 0 shapes.
Niche 11: Parsed 400 shapes; skipped 0 shapes.
Niche 12: Parsed 400 shapes; skipped 0 shapes.
Niche 13: Parsed 400 shapes; skipped 0 shapes.
Niche 14: Parsed 400 shapes; skipped 0 shapes.
Niche 15: Parsed 400 shapes; skipped 0 shapes.
Niche 16: Parsed 400 shapes; skipped 0 shapes.
Niche 17: Parsed 400 shapes; skipped 0 shapes.
Niche 18: Parsed 400 shapes; skipped 0 shapes.
Niche 19: Parsed 400 shapes; skipped 0 shapes.
Niche 2: Parsed 400 shapes; skipped 0 shapes.
Niche 20: Parsed 400 shapes; skipped 0 shapes.
Niche 21: Parsed 400 shapes; skipped 0 shapes.
Niche 22: Parsed 400 shapes; skipped 0 shapes.
Niche 23: Parsed 400 shapes; skipped 0 shapes.
Niche 24: Parsed 400 shapes; skipped 0 shapes.
Niche 25: Parsed 399 shapes; skipped 0 shapes.
Niche 26: Parsed 400 shapes; skipped 0 shapes.
Niche 27: Parsed 400 shapes; skipped 0 shapes.
Niche 28: Parsed 4

In [12]:
def process_slides(slides_adjustment, pdf_filename="./Atherosclerosis_Atlas/Proteomics/Lumina_Periphery_Score.pdf"):
    with PdfPages(pdf_filename) as pdf:
        
        for slide_info in slides_adjustment:
            slide = parse_spatial_niches(base_path="./Atherosclerosis_Atlas/Proteomics/Spatial/XML/", base_filename=slide_info['slide'], verbose=1)
            
            fig, ax = plt.subplots(figsize=(10, 8))
            fig.suptitle(f"Slide: {slide_info['slide']}", fontsize=16, fontweight='bold')
            ax.set_title("Niche Distribution", fontsize=14)
            state = {'cl_centroids': {'idx': [], 'x': [], 'y': [], 'color': [], 'size': [], 'label': []}}
            selector = CentroidSelector(slide, state)
            selector.render_centroids(ax)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            
            fig, ax = plt.subplots(figsize=(10, 8))
            fig.suptitle(f"Slide: {slide_info['slide']}", fontsize=16, fontweight='bold')
            ax.set_title("Lumina to Periphery Score", fontsize=14)
            Lumina_Periphery_Score(slide, x_adjust=slide_info['x_offset'], y_adjust=slide_info['y_offset'], num_sectors=slide_info['sectors'], ax=ax)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

process_slides(slides_adjustment)

All: Parsed 28522 shapes; skipped 0 shapes.
Niche 1: Parsed 400 shapes; skipped 0 shapes.
Niche 10: Parsed 400 shapes; skipped 0 shapes.
Niche 11: Parsed 400 shapes; skipped 0 shapes.
Niche 12: Parsed 400 shapes; skipped 0 shapes.
Niche 13: Parsed 400 shapes; skipped 0 shapes.
Niche 14: Parsed 400 shapes; skipped 0 shapes.
Niche 15: Parsed 400 shapes; skipped 0 shapes.
Niche 16: Parsed 400 shapes; skipped 0 shapes.
Niche 17: Parsed 400 shapes; skipped 0 shapes.
Niche 18: Parsed 400 shapes; skipped 0 shapes.
Niche 19: Parsed 400 shapes; skipped 0 shapes.
Niche 2: Parsed 400 shapes; skipped 0 shapes.
Niche 20: Parsed 400 shapes; skipped 0 shapes.
Niche 21: Parsed 400 shapes; skipped 0 shapes.
Niche 22: Parsed 400 shapes; skipped 0 shapes.
Niche 23: Parsed 400 shapes; skipped 0 shapes.
Niche 24: Parsed 400 shapes; skipped 0 shapes.
Niche 25: Parsed 399 shapes; skipped 0 shapes.
Niche 26: Parsed 400 shapes; skipped 0 shapes.
Niche 27: Parsed 400 shapes; skipped 0 shapes.
Niche 28: Parsed 4